# Phase 2: Semantic Search and Retrieval (LangChain Edition)

This notebook demonstrates how to search through our vectorized documents using **Semantic Search** via LangChain's vector store abstraction.

Unlike keyword search (which looks for exact word matches), semantic search understands the *meaning* and *context* of your query by comparing vector embeddings.

In [6]:
import os
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

load_dotenv(dotenv_path="../.env")


True

In [7]:
print("LangChain modules loaded.")

LangChain modules loaded.


## 1. Connect to the Vector Store

We'll initialize our `VectorRetriever` which connects to the existing ChromaDB database using LangChain's `Chroma` class.

In [8]:
# Load environment variables from .env file

# Model & Storage Settings
EMBEDDING_MODEL_NAME = os.getenv("EMBEDDING_MODEL_NAME", "all-MiniLM-L6-v2")
COLLECTION_NAME = os.getenv("COLLECTION_NAME", "pdf_chunks")
CHROMA_DB_PATH = os.getenv("CHROMA_DB_PATH", "../data/vector_store")

# Initialize Embeddings Model
embedding_function = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL_NAME)

# Connect to ChromaDB
vectorstore = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embedding_function,
    persist_directory=CHROMA_DB_PATH
)

count = len(vectorstore.get()['ids'])
print(f"Current collection size: {count} chunks")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Current collection size: 8938 chunks


## 2. Performing a Query

Let's ask a question. The system will:
1. Convert your question into an embedding vector.
2. Calculate the distance between your query vector and all stored vectors.
3. Return the `Document` chunks with the smallest distance.

In [9]:
query = "What is the main goal of smart contract fuzzing?"
results = vectorstore.similarity_search_with_score(query=query, k=3)

print(f"Search results for: '{query}'\n")

for i, (doc, dist) in enumerate(results):
    print(f"--- Result {i+1} (Distance/Score: {dist:.4f}) ---")
    print(doc.page_content[:500] + "...\n")

Search results for: 'What is the main goal of smart contract fuzzing?'

--- Result 1 (Distance/Score: 0.6930) ---
A. Smart Contract Fuzzing
Techniques. To address C1, we introduce the first Formal-
ization of MEVul and an Automatic Detector. MEVuls are The fuzzing method is an automated testing technique that
challenging to detect due to their lack of a standard definition, involves providing random inputs to a program in order to
so we formalize them into four common types of financially generate real test cases for smart contracts. It has become...

--- Result 2 (Distance/Score: 0.6930) ---
A. Smart Contract Fuzzing
Techniques. To address C1, we introduce the first Formal-
ization of MEVul and an Automatic Detector. MEVuls are The fuzzing method is an automated testing technique that
challenging to detect due to their lack of a standard definition, involves providing random inputs to a program in order to
so we formalize them into four common types of financially generate real test c

## 3. Understanding Relevance

In vector search, a **lower distance** generally means higher similarity (depending on the metric). Let's see what happens with a completely unrelated query.

In [10]:
# Try a completely unrelated query
unrelated_query = "How do I bake a chocolate cake?"
unrelated_results = vectorstore.similarity_search_with_score(unrelated_query, k=1)

print(f"Unrelated query distance/score: {unrelated_results[0][1]:.4f}")

Unrelated query distance/score: 1.7645
